# Lab 8.1 &mdash; Specialist Agents (Separation of Concerns)

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Model a specialist as a role + a small, focused tool set
- Build billing and tech agents as real create_agent agents
- Run one specialist for real and read the trace

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it and read what happened**. The multi-agent *mechanics* (routing, shared state, voting, critique, synthesis, observability) are **real Python you build and run**; the **specialist agents and the assembled chatbot are real `create_agent` agents** that really answer. Finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is a working team and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langgraph`) and, for the specialists, a **real hosted model** &mdash; `ChatGroq(model="openai/gpt-oss-20b")` with your `GROQ_API_KEY` from `.env`. If the key is missing, the live cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; Specialist roles: separation of concerns](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (Day-4 provider)

WORK = "/tmp/biaa-lab-08-01"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is present. The live specialist cells self-skip when it is absent."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
# gpt-oss-20b is verified; do NOT use llama-3.3-70b-versatile (it 400s through create_agent).
# One shared model is fine -- each specialist differs by its system_prompt + its own tools.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY loaded | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
A multi-agent system is **decomposition applied to agents** (deck slide 5). The payoff comes from each
agent being **good at one thing**: a **focused role**, a **small tool set**, and **clear boundaries**. A
**billing** specialist handles charges/refunds with billing tools; a **tech** specialist troubleshoots with
tech tools. In LangChain each specialist is a **`create_agent`** &mdash; the SAME building block from Module
6/7, but scoped to one job. Small prompts get followed; small tool sets keep selection accurate; and you can
improve one specialist without touching the other.

In [ ]:
# A specialist = a role + the tools it owns, wrapped as a real create_agent.
print("we will build two real specialists: billing (lookup_invoice) and tech (known_issues)")

## Build it
Complete `build_specialist` so it binds **this role's own tools** to the shared `llm` with `create_agent`.
The two `@tool`s are already written for you (they catch their own errors and return a string).

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent

# The chatbot's context sources: invoices (order 4471 has a DUPLICATE charge) and known issues.
INVOICES = {
    "4471": [{"amount": 50, "date": "Jul 01"}, {"amount": 50, "date": "Jul 01"}],
    "5090": [{"amount": 30, "date": "Jul 02"}],
}
KNOWN_ISSUES = {
    "crash": {"bug": "BUG-231", "fix": "update to v4.2"},
    "login": {"bug": "BUG-118", "fix": "reset your password"},
}

@tool
def lookup_invoice(order_id: str) -> str:
    """Look up the charges on an order by its id, e.g. '4471'. Use for billing / charge / refund questions."""
    charges = INVOICES.get(order_id.strip(), [])
    return str(charges) if charges else "no charges found for that order"

@tool
def known_issues(symptom: str) -> str:
    """Look up a known technical issue by symptom keyword, e.g. 'crash' or 'login'. Use for tech problems."""
    for k, v in KNOWN_ISSUES.items():
        if k in symptom.lower():
            return v["bug"] + ": " + v["fix"]
    return "no known issue matched"

def build_specialist(tools, role):
    """A specialist = the shared model + its OWN small tool set + a role system prompt."""
    return create_agent(llm, ___, system_prompt=f"You are the {role} specialist. Use ONLY your own tools; answer in one sentence.")   # TODO: bind this role's tools

try:
    # These run offline -- they only inspect the tools (no model call yet):
    print("billing tool:", lookup_invoice.name, "->", lookup_invoice.invoke("4471"))
    print("tech tool   :", known_issues.name, "->", known_issues.invoke("the app keeps crashing"))
    print("build_specialist ready:", callable(build_specialist))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Build both specialists, then run the **real** billing specialist on a billing ticket. Read the trace: it calls *its own* tool and answers in its lane.

_This calls the real `openai/gpt-oss-20b` via Groq. If `GROQ_API_KEY` is unset the cell prints how to set it instead of crashing. Multi-agent runs make several model calls &mdash; keep live runs light on the free tier._

In [ ]:
if not groq_ready():
    print("GROQ_API_KEY not set -- add it to .env (free at console.groq.com), then re-run this cell.")
else:
    try:
        billing_agent = build_specialist([lookup_invoice], "billing")
        tech_agent    = build_specialist([known_issues], "tech")
        print("billing agent:", type(billing_agent).__name__, "| nodes:", set(billing_agent.nodes) - {"__start__"})

        # Run the REAL billing specialist on a billing ticket and read its trace:
        result = billing_agent.invoke(
            {"messages": [("user", "I was charged twice for order 4471 -- is that right?")]},
            config={"recursion_limit": 8})
        print_trace(result)
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- Each specialist is a real **`CompiledStateGraph`** with **`model`** + **`tools`** nodes &mdash; the same shape as any `create_agent` agent, scoped to one job.
- The trace shows the billing specialist calling **`lookup_invoice`** (its own tool) and nothing else &mdash; that focus is the point of specialisation.
- A tech ticket would engage `tech_agent` instead. Deciding *which* specialist gets a ticket is the **supervisor's** job &mdash; next lab.

## Your turn (open task &mdash; no grader)
Run the **tech** specialist for real on *"the app keeps crashing on login"* &mdash; read its trace and confirm
it calls `known_issues`, not a billing tool. Then give one specialist the *other's* tool and re-run: **what
good looks like** is that a focused specialist stays in its lane, and blurring the tool sets makes tool
selection worse. Sharpen a docstring if a specialist ignores its tool.

---
### Key takeaway
Each specialist is a real create_agent with a focused role, a small tool set, and clear boundaries. That separation of concerns is the whole payoff of multi-agent -- next, a supervisor decides who handles what.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>